### Imports

In [10]:
import os
from dotenv import load_dotenv
from langchain_ollama import ChatOllama
from deepeval.test_case import LLMTestCase
from deepeval.metrics import HallucinationMetric
from deepeval import evaluate, metrics
from deepeval.dataset import EvaluationDataset, Golden
from deepeval.test_case import LLMTestCase
import pandas as pd
from deepeval.models import OllamaModel
from pathlib import Path


# Walk up from notebook dir until we find the project root containing .env.local
_dir = Path.cwd()
while not (_dir / ".env.local").exists() and _dir != _dir.parent:
    _dir = _dir.parent
env_path = _dir / ".env.local"
print("Using:", env_path, "exists:", env_path.exists())

load_dotenv(env_path, override=True)

CLOUD_MODEL_BASE_URL = os.getenv("CLOUD_MODEL_BASE_URL")
LOCAL_MODEL_BASE_URL = os.getenv("LOCAL_MODEL_BASE_URL")
OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")

Using: /Users/michelecandolfo/Documents/workspaces/DeepEval/ai-engineering-portfolio/.env.local exists: True


### Initialise the Judge

In [2]:
judgeModel = OllamaModel(
    model="qwen3.5:cloud",
    base_url=CLOUD_MODEL_BASE_URL,
    temperature=0.0,
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},   
)

### Initialise the Candidate

In [3]:
candidateModel = ChatOllama(
    base_url=CLOUD_MODEL_BASE_URL,
    model="gpt-oss:20b-cloud",
    temperature=0.3,
    headers={"Authorization": f"Bearer {OLLAMA_API_KEY}"},  
)

###  Creating Test Data for Test Cases/Goldens

In [4]:
test_data = [
  {
    "input": "What is unit testing in software development?",
    "context": [
      "Unit testing focuses on testing individual functions or components of the software in isolation.",
      "It is typically automated and performed by developers.",
      "The goal of unit testing is to ensure that each small part of the program behaves as expected."
    ]
  },
  {
    "input": "What is the main purpose of integration testing?",
    "context": [
      "Integration testing is used to verify that different modules or services of an application work together correctly.",
      "It comes after unit testing and before system testing.",
      "It focuses on interactions and data exchange between integrated units."
    ]
  },
  {
    "input": "Explain what regression testing is used for.",
    "context": [
      "Regression testing is performed after modifications to ensure existing features are not broken.",
      "It can be automated using test suites that cover critical functionality.",
      "Regression testing helps maintain software stability across releases."
    ]
  },
  {
    "input": "What is exploratory testing?",
    "context": [
      "Exploratory testing involves simultaneous learning, test design, and execution.",
      "It relies on tester creativity and domain knowledge rather than predefined test cases.",
      "It helps uncover issues that scripted testing might miss."
    ]
  },
  {
    "input": "What is performance testing?",
    "context": [
      "Performance testing determines how a system performs under a particular workload.",
      "It includes subtypes such as load testing, stress testing, and endurance testing.",
      "Its purpose is to identify performance bottlenecks and ensure system stability."
    ]
  }
]

In [5]:
goldens = [Golden(input=d["input"], context=d["context"]) for d in test_data]
dataset = EvaluationDataset(goldens=goldens)

### Optional: Push the Goldens Data Set to Confident AI (for reuse with different LLMs)

In [6]:
dataset.push("Hallucination Dataset")

✅ Dataset successfully pushed to Confident AI! View at 
]8;id=616290;https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/datasets/cmpcbhy5q0008lq131642w7eo\https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/datasets/cmpcbhy5q0008lq131642w7eo]8;;\

### Optional: Pull the current dataset and convert it into LLMTestCases
##### This is only needed if you already have a dataset in Confident AI and want to evalute different LLMs with it

In [ ]:
#dataset.pull(alias="Hallucination Dataset", auto_convert_goldens_to_test_cases=True) 

/Users/michelecandolfo/Documents/workspaces/LLM Course/myenv/lib/python3.12/site-packages/rich/live.py:256: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

### Add actual output from the candidate LLM to the Goldens based on input and context

In [7]:
for g in dataset.goldens:
    ctx = "\n".join(g.context) if getattr(g, "context", None) else ""
    prompt = f"""Answer ONLY using the CONTEXT below. 
If the answer is not fully supported by the CONTEXT, reply exactly: "I don't know".

CONTEXT:
{ctx}

QUESTION:
{g.input}

ANSWER:"""
    response = candidateModel.invoke(prompt)
    g.actual_output = getattr(response, "content", str(response))

#### Convert Goldens to LLMTestCases for Evaluation via DeepEval

In [8]:
test_cases = [
    LLMTestCase(
        input=g.input,
        context=g.context,
        actual_output=getattr(g, "actual_output", None)
    )
    for g in dataset.goldens
]

### Define metric

##### The hallucination metric uses an LLM to identify contradictions between the actual output and the provided context, treating the context as ground truth.

<img src="images/Hallucination.png" alt="Hallucination" width="600">

##### The final score is the proportion of contradicted contexts found in the actual output.


| Score Range | Meaning                       | Takeaway                  |
|-------------|-------------------------------|---------------------------|
| 0.0–0.2     | 🟢 Low hallucination          | Output stays on-context   |
| 0.2–0.5     | 🟡 Some unsupported claims    | Review/adjust prompts     |
| 0.5–0.8     | 🟠 Many contradictions        | Add/strengthen context    |
| 0.8–1.0     | 🔴 Mostly hallucinated        | Block/redo generation     |

In [11]:
metric = HallucinationMetric(model=judgeModel)

### Execute evaluation

In [12]:
results = evaluate(test_cases=test_cases, metrics=[metric])

✨ You're running DeepEval's latest Hallucination Metric! (using qwen3.5:cloud (Ollama), strict=False, 
async_mode=True)...

Output()



Metrics Summary

  - ✅ Hallucination (score: 0.0, threshold: 0.5, strict: False, evaluation model: qwen3.5:cloud (Ollama), reason: The score is 0.00 because all output statements are factually aligned with the context, and no contradictions exist., error: None)

For test case:

  - input: What is exploratory testing?
  - actual output: Exploratory testing involves simultaneous learning, test design, and execution. It relies on tester creativity and domain knowledge rather than predefined test cases. It helps uncover issues that scripted testing might miss.
  - expected output: None
  - context: ['Exploratory testing involves simultaneous learning, test design, and execution.', 'It relies on tester creativity and domain knowledge rather than predefined test cases.', 'It helps uncover issues that scripted testing might miss.']
  - retrieval context: None


Metrics Summary

  - ✅ Hallucination (score: 0.0, threshold: 0.5, strict: False, evaluation model: qwen3.5:cloud (Ollama), reason: 

⚠ WARNING: No hyperparameters logged.
» ]8;id=105315;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Done 🎉! View results on 
]8;id=145231;https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmpcbpxpx0008s61394nuoutf/test-cases\https://app.confident-ai.com/project/cme1cbz3902nx419rccdjpuxv/test-runs/cmpcbpxpx0008s61394nuoutf/test-cases]8;;\

#### Display results in a pandas dataframe

In [13]:
rows = []
for tr in results.test_results:
    for m in tr.metrics_data:
        rows.append({
            "test_case": tr.name,
            "input": tr.input,
            "expected_output": tr.expected_output,
            "actual_output": tr.actual_output,
            "metric": m.name,
            "score": m.score,
            "threshold": m.threshold,
            "success": m.success,
            "reason": m.reason,
            "evaluation_model": m.evaluation_model,
            "evaluation_cost": m.evaluation_cost,
        })

results_df = pd.DataFrame(rows)
display(results_df)

,test_case,input,expected_output,actual_output,metric,score,threshold,success,reason,evaluation_model,evaluation_cost
0,test_case_3,What is exploratory testing?,None,Exploratory testing involves simultaneous lear...,Hallucination,0.0,0.5,True,The score is 0.00 because all output statement...,qwen3.5:cloud (Ollama),0.0
1,test_case_0,What is unit testing in software development?,None,Unit testing focuses on testing individual fun...,Hallucination,0.0,0.5,True,The score is 0.00 because all output claims al...,qwen3.5:cloud (Ollama),0.0
2,test_case_1,What is the main purpose of integration testing?,None,To verify that different modules or services o...,Hallucination,0.0,0.5,True,The score is 0.00 because the actual output al...,qwen3.5:cloud (Ollama),0.0
3,test_case_2,Explain what regression testing is used for.,None,Regression testing is used to ensure that chan...,Hallucination,0.0,0.5,True,The score is 0.00 because the actual output fu...,qwen3.5:cloud (Ollama),0.0
4,test_case_4,What is performance testing?,None,Performance testing determines how a system pe...,Hallucination,0.0,0.5,True,The score is 0.00 because the actual output al...,qwen3.5:cloud (Ollama),0.0


#### Evaluate the results and add a suggestion for improvements

In [14]:
with pd.option_context("display.max_colwidth", None):
    failing = results_df[results_df["success"].astype(str).str.lower().eq("false")]
    display(failing)

,test_case,input,expected_output,actual_output,metric,score,threshold,success,reason,evaluation_model,evaluation_cost
